In [ ]:
import pandas as pd
import datamol as dm
from tqdm import tqdm
from pathlib import Path
import logging
import warnings

warnings.filterwarnings("ignore")

logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO,
    handlers=[
        logging.StreamHandler(),
        # logging.FileHandler("preprocess_smiles.log"),
    ],
)
tqdm.pandas()
dm.disable_rdkit_log()


In [ ]:
def preprocess_smiles(row, smiles_column="SMILES"):
    smiles_str = row[smiles_column]

    try:
        mol = dm.to_mol(smiles_str, ordered=True, sanitize=True)
        if mol is None:
            raise ValueError("to_mol returned None")

        mol = dm.fix_mol(mol)
        if mol is None:
            raise ValueError("fix_mol returned None")

        mol = dm.fix_valence_charge(mol)
        mol = dm.sanitize_mol(mol)
        if mol is None:
            raise ValueError("sanitize_mol returned None")
        mol = dm.remove_salts_solvents(mol)
        mol = dm.standardize_mol(
            mol,
            disconnect_metals=False,
        )
        if mol is None:
            raise ValueError("standardize_mol returned None")

        row["smiles_fixed"] = dm.to_smiles(mol)  # already standardized
        return row

    except Exception as e:
        logging.warning(
            f"Failed to process SMILES at index {row.name} (CAS: {row.get('CAS', 'unknown')}): {smiles_str!r} — {e}"
        )
        row["smiles_fixed"] = None
        return row


In [ ]:
# preprocess the SMILES
def preprocess_dataset(
    ds,
    smiles_column="SMILES",
    target_column="Activity",
    keep_columns=None,
    display_duplicates=True,
    save_path=None,
):

    # Apply preprocessing to SMILES
    ds = ds.progress_apply(preprocess_smiles, axis=1, smiles_column=smiles_column)

    # Rename columns
    ds = ds.rename(columns={"smiles_fixed": "smiles", target_column: "target"})

    # Convert target to int
    ds["target"] = ds["target"].replace("Inc ", -1).astype(int)

    # Handle empty/NaN/None values in smiles column
    ds["smiles"] = ds["smiles"].astype(str).str.strip()
    ds.loc[ds["smiles"].isin(["", "nan", "NaN", "None"]), "smiles"] = None

    # Report and drop NaN SMILES
    nan_count = ds["smiles"].isna().sum()
    if nan_count > 0:
        logging.warning(
            f"{nan_count} NaN/empty values found in 'smiles' column after preprocessing — dropping."
        )
        ds = ds.dropna(subset=["smiles"])

    # Number of duplicates in the dataset
    duplicate_count = ds["smiles"].duplicated().sum()
    if duplicate_count > 0:
        logging.warning(f"Number of duplicate SMILES: {duplicate_count}")
        # show the duplicated smiles
        if display_duplicates:
            duplicated_smiles = ds[ds["smiles"].duplicated(keep=False)]
            display(duplicated_smiles[["smiles", "target"]])
        # Drop duplicates
        ds = ds.drop_duplicates(subset=["smiles"], keep="first")

    # Return only the relevant columns
    columns_to_return = ["smiles", "target"]
    if keep_columns:
        columns_to_return += keep_columns
    fixed_ds = ds[columns_to_return].reset_index(drop=True)

    # Save the preprocessed dataset
    if save_path:
        fixed_ds.to_csv(save_path, index=False)

    return fixed_ds


In [ ]:
base_path = Path.cwd().parent / "data" / "ames-combined"
raw_path = base_path / "raw"
proc_path = base_path / "preprocessed"
merge_path = base_path / "merged"


# Datasets

In [ ]:
# Load molecules from SDF
mols = dm.read_sdf(
    raw_path / "ISSSTOX" / "ISSSTY_v1a_7367_02May011.sdf", discard_invalid=False
)

# Load Excel
isstox = pd.read_excel(raw_path / "ISSSTOX" / "ISSSTY_v1a_7367_02May011.xls")

smiles_list = []


for i, mol in tqdm(enumerate(mols), total=len(mols)):
    if mol is not None:
        smiles = dm.to_smiles(mol)
    else:
        print(f"Molecule {i} of {len(mols)} is None.")
        smiles = None
    smiles_list.append(smiles)

isstox["SMILES"] = smiles_list

# molecule have None SMILES
isstox = isstox[isstox["SMILES"] != ""]
isstox = isstox.dropna(subset=["SMILES"])

isstox.SMILES.isna().sum()
isstox.to_csv(raw_path / "ISSTOX_prepared.csv", index=False)


In [ ]:
ds_hansen = pd.read_csv(raw_path / "hansen_AMES.csv")
# 1: positive, 0: negative
ds_hansen = preprocess_dataset(
    ds_hansen,
    smiles_column="Smiles",
    target_column="Property",
    save_path=proc_path / "ds_hansen.csv",
)
ds_hansen


In [ ]:
ds_tdc = pd.read_csv(raw_path / "tdc_ames.tab", sep="\t")
# 1: positive, 0: negative
ds_tdc = preprocess_dataset(
    ds_tdc, smiles_column="Drug", target_column="Y", save_path=proc_path / "ds_tdc.csv"
)
ds_tdc


In [ ]:
deepk_train = pd.read_csv(raw_path / "DeepPK/ames_train.csv")
deepk_test = pd.read_csv(raw_path / "DeepPK/ames_test.csv")
deepk_valid = pd.read_csv(raw_path / "DeepPK/ames_val.csv")

# combine the three datasets
ds_deepk = pd.concat([deepk_train, deepk_test, deepk_valid], ignore_index=True)
# 1: positive, 0: negative
ds_deepk = preprocess_dataset(
    ds_deepk,
    smiles_column="smiles_standarized",
    target_column="label",
    save_path=proc_path / "ds_deepk.csv",
)

ds_deepk


In [ ]:
ds_pharma = pd.read_csv(raw_path / "pharma_ames.csv")

# # 1: positive, 0: negative
ds_pharma = preprocess_dataset(
    ds_pharma,
    smiles_column="Smiles_unify",
    target_column="value",
    save_path=proc_path / "ds_pharma.csv",
)
ds_pharma


In [ ]:
ds_isstox = pd.read_csv(raw_path / "ISSTOX_prepared.csv")
# # 1: negative, 3: positive, 2: equivocal, -1: inconclusive
# use mapping to convert 3 to 1, 1 and 2 to 0, and keep  -1 as is

ds_isstox = preprocess_dataset(
    ds_isstox, smiles_column="SMILES", target_column="OVERALL ", save_path=None
)
target_mapping = {3: 1, 1: 0, 2: 0, -1: 0}
ds_isstox["target"] = ds_isstox["target"].map(target_mapping)
ds_isstox.to_csv(proc_path / "ds_isstox.csv", index=False)


In [ ]:
# ds_mendely = pd.read_csv(raw_path / "ames_mendely_data.csv")
# # 1: positive, 0: negative, -1: inconclusive
# keep_columns = ["TA98", "TA100", "TA102", "TA1535", "TA1537"]
# ds_mendely = preprocess_dataset(
#     ds_mendely,
#     smiles_column="SMILES RDKit",
#     target_column="Overall",
#     keep_columns=keep_columns,
# )

# # treat the -1 (inconclusive) values based on the other columns
# mask = ds_mendely["target"] == -1

# neg_mask = (ds_mendely.loc[mask, keep_columns] == -1).sum(axis=1) > 3
# pos_mask = (ds_mendely.loc[mask, keep_columns] == 1).any(axis=1)

# # Default: assign 0 (neutral) to all -1 rows first
# ds_mendely.loc[mask, "target"] = 0

# # Assign positive (1) where any strain is positive
# ds_mendely.loc[mask & pos_mask, "target"] = 1

# # Assign negative (-1) where strongly negative and no positive
# ds_mendely.loc[mask & neg_mask & ~pos_mask, "target"] = -1

# # save the preprocessed dataset
# ds_mendely.to_csv(proc_path / "ds_mendely.csv", index=False)


# post Processing of datasets


In [ ]:
# Load the preprocessed datasets
ds_deepk = pd.read_csv(proc_path / "ds_deepk.csv")
ds_pharma = pd.read_csv(proc_path / "ds_pharma.csv")
ds_hansen = pd.read_csv(proc_path / "ds_hansen.csv")
ds_tdc = pd.read_csv(proc_path / "ds_tdc.csv")
ds_isstox = pd.read_csv(proc_path / "ds_isstox.csv")


In [ ]:
ds_pharma = ds_pharma.rename(columns={"target": "target_pharma"})

datasets = [
    (ds_pharma, "_pharma"),
    (ds_hansen, "_hansen"),
    (ds_deepk, "_deepk"),
    (ds_tdc, "_tdc"),
    (ds_isstox, "_isstox"),
]

ds_merged = datasets[0][0].copy()
for ds, suffix in datasets[1:]:
    ds_merged = pd.merge(ds_merged, ds, on="smiles", how="outer", suffixes=("", suffix))

# rename the smiles column to "SMILES"
ds_merged.to_csv(merge_path / "ames_combined.csv", index=False)


In [ ]:
# Merge Datasets
target_cols = [c for c in ds_merged.columns if c == "target" or c.startswith("target_")]


def final_vote(row):
    vals = row[target_cols].dropna().astype(int)
    if vals.empty:
        return pd.NA

    # Use only conclusive labels for voting
    conclusive = vals[vals.isin([0, 1])]
    if conclusive.empty:
        return -1  # only inconclusive labels present

    n_pos = (conclusive == 1).sum()
    n_neg = (conclusive == 0).sum()

    if n_pos > n_neg:
        return 1
    if n_neg >= n_pos:
        return 0


ds_merged["target_final"] = ds_merged.apply(final_vote, axis=1)


In [ ]:
# filter out rows where target_final is -1 (inconclusive) and keep only smiles and target_final columns
df_final = ds_merged[ds_merged["target_final"] != -1][["smiles", "target_final"]]
df_final = df_final.dropna(subset=["smiles", "target_final"])
df_final = df_final.rename(columns={"target_final": "target"})

# save the final merged dataset
df_final.to_csv(merge_path / "ames_final.csv", index=False)
